# Analyze button — crew pipeline

**Focus:** what happens when the user clicks **Analyze**.

**Open beside this:**
- `app.py` → `analyze`
- `careercrew/pipeline.py` → `run_analysis`
- `careercrew/crew.py` → `build_crew`
- `config/agents.yaml`, `config/tasks.yaml`
- `docs/CODE_PATH_ANALYZE.md`


## Say this (30 seconds)

> Analyze is the core. Gradio gathers resume and JD (upload or paste), runs input guardrails, then `pipeline.run_analysis` builds a CrewAI crew of six agents with MCP tools. They produce a typed package: profile, requirements, gaps, rewrites, prep plan, screening answers. If the LLM crew fails, we still return a **deterministic fallback package** so the demo never dies. The result is submitted into HITL review and shown across the UI tabs.


## Call chain

```
Gradio Analyze
   app.analyze
      resolve uploads / paste
      guardrails.validate_request
      pipeline.run_analysis
         build_crew (agents.yaml + tasks.yaml + MCP)
         crew.kickoff
         parse → Pydantic models
         OR deterministic_package (fallback)
      review.submit_for_review
      session_store persist
   render Match / Prep / Screening tabs
```


## Agents & tools (what to point at)

| Piece | Location |
|-------|----------|
| Agent roles | `config/agents.yaml` |
| Task prompts | `config/tasks.yaml` |
| Crew wiring | `careercrew/crew.py` → `build_crew` |
| Shared tool logic | `careercrew/document_tools.py` |
| MCP server | `mcp_server/server.py` — `read_resume`, `read_job_description`, `query_answer_bank` |
| Contracts | `careercrew/models.py` |
| Honesty | `data/answer_bank.yaml` (`verified: true` only) |


In [ ]:
# Chalkboard: six-step pipeline as a list (no CrewAI import needed)

STEPS = [
    ("1 Resume reader", "Structured CandidateProfile"),
    ("2 JD reader", "JobRequirements"),
    ("3 Gap analyst", "Coverage score + GapReport"),
    ("4 Rewrite coach", "Cited resume rewrites"),
    ("5 Prep advisor", "InterviewPrepPlan (seniority-aware)"),
    ("6 Screening drafter", "Answers ONLY from verified answer bank"),
]

print("Analyze crew sequence")
for name, out in STEPS:
    print(f"  {name:20} -> {out}")

print()
print("Fallback story:")
print("  crew fails / no key -> pipeline.deterministic_package(...)")
print("  still grounded in gaps + answer bank retrieval")


## Guardrails (say if asked)

- **Input:** file type/size, one resume source, one JD source, no raw job-board scrape.
- **Tools:** confinement for uploads; answer bank query only verified facts.
- **Output:** Pydantic validation; prep advice must stay grounded; screening never invents unverified claims.

File: `careercrew/guardrails.py`.


## Code checklist

| Step | File | Symbol |
|------|------|--------|
| UI | `app.py` | `analyze` |
| Orchestrator | `pipeline.py` | `run_analysis`, `deterministic_package` |
| Crew | `crew.py` | `build_crew` |
| MCP | `mcp_server/server.py` | three tools |
| HITL handoff | `review.py` | `submit_for_review` |

**Next:** `04_hitl_and_coaches.ipynb`
